# BOMBcompeTItion: reproducing our submitted network

This notebook rebuilds the network we submit (`submission.py` + `submission.pth`), from the game engine up.

**What the network is.** A 64-channel, 6-block residual CNN (about 0.59 M parameters, 1.4 ms per step on one CPU
core) that maps the fixed `(13, 9, 9)` input to 6 action scores. It was trained **only by reinforcement learning**
(PPO) against a league of opponents: no imitation data, no search at play time, no game logic inside `Net`.

**How it was trained** (sections 4 and 5):

1. *Stage 1*, 200 updates = 26 M environment steps from random weights, against league v1
   (hand-written rule bots, 40 bots of a parametrised family, noisy variants, self-play).
   Reward: win +1, draw -0.1, loss -0.4.
2. *Stage 2*, 628 more updates = 82 M steps starting from the stage-1 network, against the wider league v2
   (200 parametrised bots, a search bot, six "student" DQN networks, more noisy variants, more self-play).
   Reward: win +1, draw -0.3, loss -0.5. A draw scores nothing on the leaderboard, and the larger draw penalty is
   what makes the agent hunt instead of just surviving.

Which run to submit was decided on **held-out validation bots** that the network never trained against.

**Two ways to run this notebook** (`MODE` in section 5):

- `MODE = "restore"` (default): takes the trained weights from `bomb_best_assets.tar.gz` and rebuilds exactly the
  submitted `submission.zip` (md5 of the weights: `e3e5edae202d88beeb291b4db92aa0f7`). Runs in a few minutes anywhere, also on Colab.
- `MODE = "train"`: runs both PPO stages from scratch with the original settings. About one hour on a 64-core
  machine with one GPU; on a 2-core Colab runtime the game simulation is the bottleneck and it takes many hours.
  Reinforcement learning with many worker processes is not bit-reproducible, so a retrained network plays
  similarly but is not identical to the submitted one.

**Files.** Put `bomb_best_assets.tar.gz` next to this notebook (on Colab: upload it when the setup cell asks).
It holds the trained weights and the student networks used as opponents. Without it, `restore` is not possible and
`train` runs without the student opponents (5 % of league v2).

**Local results of the submitted network** (argmax policy through `NeuralPolicy`, exactly as the server plays it;
every map from both sides; maps never used in training):

| Validation opponent (never trained against), 200 games each | W | D | L |
|---|---|---|---|
| `zoo.val_smart` | 64 | 134 | 2 |
| `zoo.val_hunter` | 10 | 181 | 9 |
| `zoo.val_tricky` | 62 | 129 | 9 |
| `zoo.param:9000` | 195 | 5 | 0 |
| `zoo.param:9001` | 198 | 2 | 0 |
| `zoo.param:9002` | 196 | 4 | 0 |
| `zoo.param:9003` | 174 | 25 | 1 |
| `zoo.param:9004` | 146 | 50 | 4 |
| `zoo.param:9005` | 173 | 27 | 0 |

Mean 67.7 wins per 100 games, 1.4 losses per 100.
On a second set of held-out test bots that was not used for any decision (100 games each): 44, 34 and 31 wins against three rule-based bots, 69 to 91 wins against six baseline DQN networks, 1.8 losses per 100 games overall.
Against opponents that dodge well most of the remaining games are 300-step draws.

## 1. Install

The course's Install cell, unchanged: it unpacks the game engine (`bombcompetition`) and the example bot.

In [ ]:
#@title Install the game engine { display-mode: "form" }
import base64, importlib, io, random, shutil, subprocess, sys, tarfile
from pathlib import Path

ENGINE = "H4sIAMuP/moC/+19CYAcVZnwhFPbcInK4cGjA6Q66XR6ZnJAh4bMZCbJaDIzzkxAbNue6u7qmWJ6qpqqmosQEXARFhFQV05ZRBBlEfFcRHTDtbouh6IIeKyIrrg/rMCigqiw3/Gq6lV3T4L7sxHXLiXTVfWO7x3f+b7vq6I9WSzZkzXDMz3TtpYXCqZleoVCqjbX9nJdabhWrVhBf+Gq/5te0R684+ftnR0rVreJdNsuuKZcT3eg+7a/zqvi2JMiZVjTwpys2Y4numFDGE6vNZ0UE2a1WnCMGd0px7jcmD5p+AV7ejeNdA0nxeauQfhn4IRe+D20ZRP+6SrhXkqKHkP3xpNiA9SCd1NVw02KYlV3vULJqFbhpmqM6dWCTsXh1jGtsYJdkZ1N6l5p3O9tyCjZTjkphvWK0W17UNXWy4WaXTVLc0lRq+pzBSovf7uGYxpu0FDN9dtxdKtsTxagjD0FrchbB2GL/TWuf7EO/3XXNTx3+aa+db39w73LirQbqjBLqcny/xf+r5oX/zvSq+vxf1X7ypUt/N8V1+a+EbHJLBmWa8Ri6+zanGOOjXtCKyVEB6yMWCZqjuEalifW2WXDEQOWIQa9ObFppCcWGzScSdN1Yd8I0xXjhmMU58QY4JRnAKpWHMMQdkWUxnVnDAiAZwvdmhM1w3Ghgl30dGA11pjQRQn6jUFJbxyace2KByTHgMJlAdvRLpk6tCfKdmlqEgDRcZ+KigkYKzRv3BDxYVkjnqBOyoZejZmWwHf+KzFjeuOA8QIG4zmmpE+mVapOlREG/3XVnDRlD1idJsONQaNTLowA4UyKSbtsVvCvQcOqTRWrpguEp2xi08UpDx66+JBmNYnjWG47wgWSF4MWgC4JGmsIHZVB0Gs4oZ6cIhefzIwDBYuMxASyNuVY0KVBdco2TBn1eLJR8vAJFq/Y1ao9g0Mr2VaZcNvNxGIj8Eov2tMGjYXX2rI9AJVBwAWohasqX7njerUqioacMOgXpldXhuNg94BIlmfqVYGEFvurH2YK+t/YK4YH1o+c2DXUK/qGxeDQwAl9Pb09It41DPfxpDixb2TjwJYRASWGuvpHThID60VX/0nibX39PUnR+47Bod7hYTEwFOvbPLiprxee9fWv27Slp69/g+iGev0DsKH7YFtDoyMDAjuUTfX1DmNjm3uH1m2E267uvk19IyclY+v7RvqxzfUDQ6JLDHYNjfSt27Kpa0gMbhkaHBjuhe57oNn+vv71Q9BL7+be/pEU9ArPRO8JcCOGN3Zt2oRdxbq2APRDCJ9YNzB40lDfho0jYuPApp5eeNjdC5B1dW/q5a5gUOs2dfVtBj7ZtblrQy/VGoBWhmJYjKETJ27sxUfYXxf8f91I30A/DmPdQP/IENwmYZRDI0HVE/uGe4EBD/UN44SsHxrYnIzhdEKNAWoE6vX3cis41SKyIlAE77cM9wYNAp/v2gRtDWNlHKJfOBVra13/V/k/SWmpmjX2ssn/8/P/9KrV7XX8f0XHqo4W/98V17mD/Rv2iR2KuLxP38aeIfi7P/73qr3g38OeeVGHP3sV37ahp+1F/N9VX7vuw21tC6y+nq6Ro88/fzhrnzn4uux9z33n/KNXfv99377E2/+61D/ftc/eT9vef61OP/3GJ+7/p8P3/fzU4alNj669f+PGG06JJ8/f/lRmxZNPXzn+s0tvvnjrvX+cffa1fzjr2u9fd3X2vOmfd3zy0f3PveWyvheEtaXniNLxv/qOsE4pztR+/uK+P7x3y0W3bHzDXVf+271XjV/8218ufPTIVxX3+f0pFxx3+nsO6BPn5Nra7uhf0HbPgwd9+rRvHLtg8Kp/OOSCf1+W/NJxDzyw8Me3HfD43+3Vlv7Yhr6r3lFdu2Hy5hMSHedNX/PPux0813H+6EMJ5/iP7HHoM4efc/FIZbj08JNdpeN+/pNbLtM//Rtz88L0Ex/5x9oxu7d984/7n/2xMz/4rZ9+/662R49MP/H4z3OPvP6mmelle37lsNvbB7Q3vPF15+3zpmfe96P9fvrO2DkXP/L7n318U+ybS9dcuv7WngeHZidu6P5+/Kl//MFpn/jtz95ifNf47CmvuefZw+798NmzD03f9G7z7944etJ+P9zX3nJy1w3dt+6513e+ccJz17fdkR27+6nXLHruqmXPn/npE3DB+nr7e27sHj3zfxn/4XFBf5kIwM7wf2Xnqnr87wSVoIX/r1j8P9DH//b+O9P7X3HFjZvP++Kba+PPtN3/7St6Xn/uUfss23NFYnCJ99m1Jz7d9tYl1/dcffXHn907/kjmuAv3Tt4zt/zLzx693vnpb+6evfXLnQ9dmP3NE6ML3ntF9vTL/999P/xuT+c/nvPMs3+fXfDCfgvEYwdd7ExtW/we64Cxj1546YGPfvmjB2++b9HdHzj2vgVtPzu6ePBer3/jbxaec+ojfT17xLuvuObxzEeMQ7532m4//tg5N3xir6N63z7+tzfvvfcBiy9ZcOH0l14Y/t6zX9v3gs+sX9G1ZfXs3LbOuy7/9+73u/tu+fEvb/nt9uvGn7R32/24426vfOXbv/uXbxz0gwcOHbhgv7YLj3zwHQ9sHi0esmn1wML3/njq1KMfmz777tvWnHP8h2c/tOeLN30y8/jq6y86430vvubDC0fXLjxtdr//BST9M/B/xP/irsL/Fe2N+N/ewv9XLv6/wcf/j/fv0b7/tq23ie9/6ANnvavj9wtuuO1dZ/5gjy/sdd5t1Ved1HvvUwsOdmK3Lb5q8KuvvnL097tffmHnU3suvfCsr9w7+cNjPnjxiPf8vz3++BfvGV03/b0vfG3Ptv965qnq9fs9ab/2njN6//DCcz844JEX929zTp1OHXTxg48+9oZTKq++9JjDrZPP/c6vj7397m8+86a29AcXna0Pfu7t9733U3tf2QNa5J19t9y47qSPdt2z+Ns/PvebmZNHh173N588Z+tNB51y5iFt5sX/cHzvtp88/tRD4z0HjkyddfGDq4dvfffdU0c/NvjA+TdO7Hfa3/72t0vSj/xs0Q0HuR/7+1tPvzp/41O/nt277STn/C0/3uOoj97mPXNn20Pvvu79rjXxzTfdetZznecuPump/Sau/8J7PnHoIRuf+6dbv6gfNf31r//LXxAJmAf/K1XbdgrtLw8B2Bn+d3Sk6/F/ZbrF//+8+L8b/HvhReOrmuF/Wy/iv3ZRqTy6/6jYM33RNad+e/y5tVefvfn95z741Bn6d39x0smXLbx81R3/qR/xphsuX9X53O9un+sZH33uvTecffA+2mjtzjtGH2xre/Hh12z57s1H3PKXxTH/KvR/xv+OXYT/nasb8L+9Zf9/BeP/SAP+j9yx3Rz8+dCizevOffzIcqpz9933fssXcl+78ls9Tz/0wgvTf/zc99//E/uQgw8++NWfvPbTLxz/+dXxhXeN7b7H9nvObrt91VULPnTW69ue+/sDzc+d95n/apGDVwr+d+4a/G9f1dmI/y3+/wrGf60e/y+bPvSdv9p+6QfOeni/N574+f0/oQ/+Xt/DbGs7/PLdF55z6/NPt3D6LxH/V+wq+X9FI/53tvD/lYv/fU34/3tPfqTyUM9lZ79u8+uLK2ud1W89eGvljS8eW3jm0Dvfsr54/9DXf/Glcx/80sba/Vc8PvNI9erq2r2O2Htp24XGwp9ftfLaVItAvALxf+Uuwv/2Jvp/y//vFYz/2Sj+H/DpD16uH/jqQ28+ZNW149ev/XX6tf9xhHX25sIhf9j47ba973xeHx/f/wept37g7APe8tae3dqeTe+1aM/HfvyjFs6/YvEfnSgN52U6AdwZ/gO21+P/6hUt+f/Pi/+K/f9JxPbZK679cPu+Z67dv/f4973tHReN7vdvt/1t95Ojjx75qi/9dH/n1ZOfumnV6/b45ZEPr//g595QWXXHE1cf1f+Fm178p4cvfXPbCwsHrDvffvXFd9/xUO6FtWe+/21jd/5w22VfPW/kX3vectse77xw0chFlx1x3EHvOWOBuHvJ8Oc2XHfgadkb29p+f3//USu3XPNk7Igrblj9i5NuH7r8S2vPvrZmLjj94YsPfH7DY9Xb7r/sDW2Xd1+y5bzFue98cXDqg/oT2c6BWx6obX3+m9dn9n7khU/tv+HcD36i58l7Bz7zpekbf374Pz+Wts/61dobt/9hzSWbu17/Hw/+9tr7Np1x6qP/fsqiz977/EGzT+3xo5u/+OJu73Mu2OdnT73zvX8N9GnH+F/cJfjfubpB/1+9snX+94rB/9sZ/0ujV+2fvvo1v0z8x4sbflG4ZeaAdfv0X3XdNS8e9uar3rhxsPPTZ9+0/Yj7Tv166bkPL/pG7elfLbo6f/d2uPb47sLdn//WXlrs13+3z6vef9aVH3rh9IPX/3bfdn377dvXFjeWOtsOu3n22d3O+vrnFzz99vOWveGmu2InfPan3z3tD/f+zYvfePJd29PnnHvw6VfevPWiM+C6rfukw9bte0fbJcXy6MdvuODjh3xy34l3X/KVicO7Trz2q3cduO/dP3n485q5+6UXfPc16bu++ekDD7niN6e9/0OlBYVT/uW1z7wu94GWyPHS8H9Gr1ZfLve/nZ//r2o4/29vb/n/vWLwv/H8/yg+/wf5f3T028P3n/6va+M3e09end4drk8Ahuf2TaU+/r3n37oXXGc/fP5/LkyvOi791c997PCv3dJ92blnnLGg96wLr3jzurees7xr96HHlx5y1APfKxhnvHr7is987PSP6Pdeds8e+1z6hUrn7j96zcKFC7uHDn3LQrzus54YXVvpTKe3/y7dduBYufwHICHbt1/4w4c/cMTkPmObUz9ZMLb34neWPnXp0Nu/sf19T5zmwau7xbU/PeOTh7w+/dOPdO323O1nabs/teDmBQ9VbxvrfFf/QN+L//nEA5e/UNr9Cyfaz3/2umsuGO4vPPxI9tg3z5564iNfnARIFk48dthdT+7pLLhrr5vedcvM22enS78//fBj1nxoxV6L39mZ/uPn7q/gLFzy1dM+8b1rLnv+H645+MIH7//UH5/7xe8Oajv3+jPEHV8eqr2CqU49/hvW9MsZ+vdS8H9lR2c9/+/oTK9q4f+uuOLx+Ia5SUt3zalJAYufEXP2FMXQCdsyhGuWjaQYtWs1uLO8UaHZRVcsO05wzF6CSroU5WHDP04K2ovFZKzdWNCw7uJNrEkUoYwYVEIEZSHHsDDeSBbz7IIzVsTwnoJnzHqxWKxsVNQARa3mGNMFAC4pQUsKuiljCKKbQJArVVv3MjEBF0CJkTDQhj5VxaAgbCIjlraLmXHDwilwBGKGGDc9OTo5A0mxLCwkqrZrCF1UzYpBI8e2YWxZ7DsXZyE6nqfHjuFNORYDoblTk5pWTiH8MMZsFioBiAIecR3/UcV2BAXbyGEkYOClKvDoMExTg4lNwd9EMDL668ObESKyYtRksFyCu6Ma/DODP/E9rnywF5IiDYPqwkCfdvjRzSPC1aLyMAW0dDAAntEMd1GpmCUMCKJFTshpwKkuVKyMeKlLlhQlkEaMstArHoBsTBvOnHA9o7aGWlQurgnL4cKqzYzbYgY2HiyhoGgw3aNaFOhUnOPIKo1CVDUeO0e8Gk4iEUylXFFPL+ueDiPfGud9WZi0y4Ybz4hcXLdcM54UcdifBd1x9Ll4Hu+4WKWGhVZs44Zwz/oB1pprVCtJZV/5829aHnSUTvrzyzOb9TElmMCssvuT9TOBEx3AmYFxO+I00Y8YnaU/crfghXCkQjDo1p8PunE4cJd/h63iLq8DXsKcVDuP9hNAD7WD30ER2AFuDd4gNdCoqUTKNU8FyoCb3K3pJSAPfmEMIYaybi01MlWrGpoGv3pMtwSIZmhWAptS7+Wi4lW1rTHDxVme1Ge5n1SFAvzC3wV4hds9HVYjgpDlZkuetjVes3FxEQ5YcHvGAlzPRHrtACji2Frd87AbsVS0J5qsnrzi+lh9XR94rLgtUbeKRddwpil8sUCzpYIb6cSnTQ3wRkuhGuByoc2A1ma3aenOnKbBQsGM1hWeNPzZiD6HXdL8BU6obH3YOGXKsEqGhs/qG66a04Ys12SpeTKpDM9mQiwRDSNBzG+6CrDOBXzp1i1F/dwyiYpOq2xoVSJEbwyW9SRuu4ZRziK2IZpTRGS2HvemaoajJVJ+LSiP/9R1TfxSYkUUIxvKSy4TVMMtoSkoDRtyq0KMcOASWMnQQ9jGiCVTz0FzEfIQlPTpOJQM+gTuFbyX8f3wOgdkLZ2vf5GbNPLJ4KYdGCE8gNJACTUJVpJuIqRKC/riGiqGS0Yg4aFRLpHth4V2ADGTJngr2XWEdCl8KwKhyrzCpjyMprUoghnojWkx2LRdE+JYIPRBSdOq2MRiuA3YrfwjGWKAR9tfaQHeIRuoe0ePEtvqdwUBKBmGAleS56CM3AF5o2V7kbcIl7rBkbTTlCibRfbAYpoW7JeEMCtNGEdW5ZbCqAIdlIKdUrUVXvrn0P9oK7zMCuCO9b+OzhUdDfrfis50S//bRfofC5iI+cRoDGvMtDBWv1cvjbPEXARlQYp4LtJpAaI06QhY3jMnjUysPSW6BzZ3i7JjAyvVWViypSpFopoGlSZtkFxGkeUS7x/FDAFYwsUWbeD/iVSsIyU228jMMdlBsWqXJgyW13VKGUMVZbKCqBazRngzdgDmpD2N2Q+AQtukyRIMNBBY8LlUrDMl1oMABtqdWZoQZRDfUqDgENgAS1oYs7UqyviU6kCUHNulfAajjl42pwBymcHG9YAfqQDGiGt7HnYOCpCEkKE2UdWEWQWxA/qSqp7MVEH9UdIFVCpdqVWuwTom3iFNx7YD/U2yQaSw6v1ilweBqk9NB9UZgDa9FFY90YTRj/rSLeryJJXo1TXUQtXQx6YMMcW9V8xZGBYWBiajC8uYoXaheZJbASbOnYPw51h69lvO0/KMm+WyYRGLIPhC2VS44/YMq9WG7lRNFGdhmCXd8medG7BnSFQX+piN73FEuL1gulOxFTCDpA7qzCWjezRgR9iJ6eCSw0zQknFaEdoLtJFgkzkGbBZsdGVKDE8h1LJNbdQdd0xrokA652giE1FBoy8FC5E01Clkn7jV7Rq05fBWoFeTek0gaC5vTNoyOEPaMbPHSKUZq6yeXS2KBtBlWIqVsyuT+MYSnbOdiRRp/i6iFkyJAv0amAm5o/idSTjAOUKwYQvm2vBhwqmIkcyBen5K9FWQ7Y8zxpDqD9WxZDIAG3e6K3jMUC5FuUQkxSi7bBNxAWJEtnFYKssWLJRXjYqHqhTP3WggcI+mYqq1iLcTW39Q2yY7hxEkbgoecQnDmpr0X/VZXi/cBi3B79oc2pysWiy2jtVEko5yJuqq8E9eiEVCc+wZzOlShQ0OPwHnTV4/GGgstrlrEKppbAlY5F/xpHzQlUotSsF/wYOU/J/6IBV90KRKqr7KoiZVusMHChyhLYjTbWlyGqRUNjzSdZLwxcstOJZ2+tkzcGI/3HTQzabe9SNw00k3nKwkK1bQHVHzrFgJvXCmL5wNDeV3mC5tWbv84f9No2FM/vD/phMJmudygWa6XMC5xvQyUtaPUdYwbJdHkNoy6OcOSyGYwQ2CGdwQmDj4tcGW0GBLnGpY2RFnCuRGnhTiajwVnOxLSsg513OSAiYWNQxcZIBw8aLFhIpJsTi1WJDvG/zsWrx8cfdilXasAaLhoFVyEqQGzIcT4C+Xcv0+1pFNAP/NR+0u2BsgpeOYyFpws/ndUAfUFJJR3wy0EiswZWHEQPrHSZOIyntIGd2QeayhTEYBJWaSAuTW8RD1a8DEzGLVCLrhMthVA5QkuAO2IL/6H1F+oWFTQDKrM2giJr4zmiTab0lbILFSf6gd2GvIBYlRYhf0t2w6Bu8ZMsj5AoRftxPrMkGvBfKAkEmcijbqclLA4A2BdMmv205rAoNGfssTVjEdmC00HBJp8ySh44VWiH7d1LWn07RcCgvJSF5BiZUmMf1VyCaQqDKvxyUkpsG8Qk6chVUTwYCpYjDgdBowgEyCaBrEva4RskXsrpKjc3Y7FWEUIzKjCFmvqPG6LQhNwkpUJdhEyjW5vkTxKywuIB+iTcHw6mPBDk6HO9g1YQHqOPlO8ZgMtJmIhdqHkq21kUXwZyAwTALfoRcoNumkilJOOxwEbojqHEpuJKTCFPAWSsjTBSVToUaIDktSS1llUluThLAZwZiu7GSyWwPmEA3IB0b5dSR7YKfQnx7ImaPYymhwdsASSlZsxR+sv9MRAJCjcgmRgUlxrj2zMh8q31hkAt/CEowZSJIZHrJoZSJ2MFwRDRvPpfPwuuyIJWKCx5JrpyclfJKIVAJSQOPPeflMg6myCNtjIvKUBpHSy2XNS6hHH0zhfKaF1qxgdgZ93QL2URrYQYLoSLvQuhOpwAzfxHre3EQe2N2UwUMr+DgDQ9al7CmpGO3gNaE+oxjXWFaU1ivHoFOxoBxpyv7ChWwGDdtskaTbOvM3YHdWdpwaoj9aE4Mf2ryhXNWwNG6l7j2tBhSA7UibUcvlgEaiWWVRnHYDU07gunm6RekG7rmtfNhYxbTKtNuATmkOSUEpeGTMaqXxBJ8+8VOiwyBbGI6OJmiGSekp3tUdx20S9LutbkDEHXFIputJw6t8Bm1oCEdqzPC0eBfm8Atvu+OqVc+3JdHwU+64XjPE4bCfgzlLhtOXwIbRkqWFVXI5klLy+RTcabzHlLeZpIgWqMMdRzeB/Z2gV6eMXsexHS3uU3FfHHBPmfJzDTLGkKKhO/aUVY5HxkE0ybQis8M5Av1bRNBsNvIEEBSpmDWnjsnlJXbrWtsZ7JV4US+rsk1GbFWqb4tLm3a4dSWOMTOJvGIxPytyiineP0uhm3y0PJknG63B9M4TqkmUHpWZu67Xq27daVIBtNICza6mmOCVp3VWSjZe2zBW3IdMof1Nqc5dtIKUNMpmyZOCHfJNrLi1bpsTwyC2kQlZANnPPebOTN/pKFJqumVf/8daxCebNLkjkOEdNQ1/Ocujb5wB0cXXOoO2k7BLYbcYUqGnSWp+GpAmZgY/2kOuRhAQP85HCOtg4C3gH69Kio9HsGWTx2h6EWLp4zIubrOtOjRloWFJQTT2GnBJfI43O1yQWoSmo0Li/25PKBsZEaWG1owIOfMPBqJgTJq06/Ckvpgi8QjRsUZtFANko72Rmka8AgGsgW3qWEfqLaRQ4eRQy8cqWJUK5NmQJMFy5mp5omCRvhoZcPgup9TETYLbVKslFVxBXqepM+JhalaizFy5wA+wlp7YyXzl1cWUDSHVApLs37XXCQw+gCIoEoJCeJNEEakAIpKkfAWJJlqTs52cPLsHhRPT1CIU8oclW8EZpNMFEgZ4OPiWldRgD8q5hhcEQz6yXxT3iyZjIfKW89028mKZr2YrUxP4eZgurWd/w5aPUsacXwHkMWyunkAuVftQKSSeLoVQ8elSwFU8cVy2fs+RWK6uIoIXthhsRsmwFc0nfOeJI+ctBJs/3WTSClxIk/WXL5+vgXDRcT1530Qpnwq83Bj+EdY85KWRccxDwpNRAgxdKzsUHtSffXH3ChtSRpkUE0xGI3RzOKIokuo3wSmNpZkBVW18mk4E1kBi+msCQyeb+tDON6+Zz7fxqdQXW0VDGFE1X6XFh4pkkVTVALk3qEYduabRR1BG9kq9NJl8lltKkpOhrhdrTs9ghWsau1cQAkfpeIjImXqMq8PqRjgIcKhbSvuEhks3FKIc1kBqZqnPWWJkNGvQy2wjdabNN4sUMGiUSFKeJDeq2RzxJbCMw9gpzL4xl63qk8WyLmZBPNeLrjaL5HUZQJ4ATY0ftOODEnK8WZ+ur62Bam043lx4ZstZ7on6k0iE3Lxo29VMU8eBOi8YhWzVk4SgzHH1T+vEsWjXgAnz9GyNpRDY4LBfdc5p0q9yvhwOkUn+/PBIFsd4WfPlnIzk0wQgilkhhKgaldCmyXp4Irpnkr5Srofbh9RshxRsqEpqteJHJM+0sio6eLRD6rcTPco27Kamc+fv90rQAR2SRTajzw89ZTp8DhusjjST+vJrkuU+vEfLeT4q+43g6Zk0ApHg59pVzC4+roOUUw7teW4qqDVk8AlIcOqFUnBwNpNULHh8+EDCV5BMnCeGRGg+S6ODwMkIeduRjBYlBEX2wmpg28WUPmZEOW2jfAJCfRLlJvpLf8KVOWXKmCLKQcTQp4rJBrBMz5hE0x0uG8OCXDtsZ2bcrBrcWqbBygLt0xsilA3CJ/lSiUx2BxQ1gcjdXB4p2SCAW3W0Ge1X2YhhTOUX3LCCeGyGigJGlcVpWWwr8oJnbGmU2JLzZoV/h8OI0ukdcoPmHAGazTR1ugvkRrJfYWcs+zd4WEV2Q4hNM0TjI5ROmiajWAMbWRqpyVgwiTpZeFqWVG2WaLBqqjqx7KmYCfhBvcwtAcYW6wcRkFm/pk/H/aYVrdp3JPNJJg0PdeLIwE5EvJQjq6ENzc0E5nO0NngGnsvR6WRUIGGA5nNSBBkfVFzyQU4qHsiNfoqheQi/b4B4SQcDdlVo0kCEtqGktGf6B4D5Jo6MKoVv5s9YR5KjZRZJCzrbvVGVreKgfZ8CeUqjGiBplnq7hjb19Q6P+KfhWjv5qM4VDflpDGgOT+OpdF2HZKX3D2jYQp9iO/z8xvdUc39M9iSbx79VIkTS900lSjm/Bys1LF1g0du2vYGPw/RRGzvg5UwZt+2ok51S2PmcSlUEQGau4gOtLO8hDbX2pOJlkWjuX0oqjeJFKs8S5JeNNPXQALdiiEqBTARokQ68PFgRwD3vP3HUM3qPTjJ9RJJIhCKkNPEHln3uDcaDYwpe1j9rz/tnH6pK0OTsYyKEu7m0QL/zkWAL5vB2RSo6ydCHx9dU3HBcwZgsaQcnIBKRQ5Icy2Vspg6PPyxpmFafREIwtK0Kg5H9his0i67NSFIntkV3TG7ntY4D/c1XP5nCzOaDKY186wrDaNikGFrWWP4MJw1P8JMiMBuRsZkco0j4QUMz9qN4SOnyENv3voo6CgVTyhJUVmwt5gjF86G8RGErTAbyjG7cI0x2PjiNInMQHd1nogIv1QbimefDpUAspr0WvmyPvmzPN6iZ3BIT9bwqGkelYiqF1Dhfb7tDoFM6kDhQ8PXIIZSc5RRPrmKaS4olVC0f+6v2/8Rvpe1a/8/0qvTq1Q3+n6ta+T93mf8niyG08qKXDHd4qAVixDGzxzQ/zAoIEp2E6tIRr/3o9O03kXsbsBZbWIZJvo9E0tEjTLdienlatzzg9mtEl/SZwfMKFAva+UCum+5XJ8XqBAo60ofDj9QDwFIsk5I7TN3urfvIX3hUpUZQqC855KIT/ZSE7ztAQ6LT3zFzWgpm9Az6psbwY4lZJQAvdDagNrORDqjpRf55SzDT83q+gTRaNeSRhu/5hrEx8/rF0bF0ydhBbGXjhxdjseG+d/bCII6JgTyNbl68AEmh0YtloiMp/F8+/ypMmnQYgy1IeitpqiwZES3qn7Xn/WY4tZjiVhFtbB6uju01MHZ+KJmGFOnzuVI+6KtYkX0kebMlfXaZZU8TUtpl/7DksG99vX0rlc+I9LYkL4CWoyeSWTVVy8nWo6jkqMppUaulRXqq5uDUSKMSmomC39jCMl4JaTlqTzTaN8PhWjDZefi3Pc+uV1ZokwShKvJEjrzZyYNh5Sw+p4RfPP/Io9sbivLQJGO1IowV6/rTDrpLwXBLes0ogLJAR1taxEbAsX2BpLMety67wiQlNWlwdMOvCybZFZpMH/aUJ2VFV9oUyBykF6t0sI6Hh8f7Ek8Tv9Dm1ozAoYI3ZgTciDUDv/UHddUNRnihzgge1Jf82ce+cDnKaN+JKD8s2peS8qQJGg5UFn8+7Qm/E3XiCJEQiEaMiikbpeNo7BJlaCpHx0IrVjTYX8Pjdfa8iw6uK2gSG6ISCbQsKs0ivuJz8tzoRu9Pcaw4OuN715VsywLChmZJKJYMTjAo2si2RQld4eYHapGQ4QjosIDMxLHxw4j4+cPyGO8CdOBizz3cMcyC7KJdnhOkOk+VJvyzdtQ4yIVZXa1qVdvRxg0mP3S30JB4BqsU+bKspsT+NXNSUlUn3x81QIcun1G4c5OTBn6ukjix9C/Sdu6t6iZCnXCHHkf+CdA8RLc9pOMNtFd9x7pBTYfJbzB5Ro5M6AHsvwgvyQdbhPC6wh6fXAK2CzbL7n6GUSuQsgjkGaee3Y8xyFbrYPblt8vP1QdUIBG61BWUkaTTBRA7FTJbBs5gemjkxQOJKcuEGpNaOgWNpFOdq0KS7jti5XJ4jJUXS2hOkHb6T+AXbWJ8FzBXPL6R7+tgCUqobchWm56x0aQ3MAh6KylPOG1IfyIz37yIfwgDOwVRWE5GI9+Q7mG0J6RROdJ6ormJVeHT9cd/5ry0LqALGDiC842ebuiXyAaieIr93maaub1Rc/mGZnJdxDu7kHcm+Uk3PekmbpoV8S4M6++ON7OmsoksHk+dbJsWBhIk1B6xMd/FudGZxbLFtF41yyRlV6RvWISGKLLpTkiI4uUOGD9K5UZZZtcbqQhKz3XGIik+atLrkWlMtpGW+cshBSkZydyK0fy/p/97wMJfXgPAjvX/9vYVDfm/Ola18n/tMv1/k63TN7CLtseHuCia4QPUJUG4NrwZVCaCU1ZfbeU/VbOYmvLMqv/0ZNe2/N9I9BrU2pReLPkS+TogzSix7zzqC72CjWqZC9Z0bxz69QsNwm3z4K95deIumXCGXMii6YfYWAtzUqjZVbM0p1XsahnDDDiJCnbG0ttczYgYuAepuGD3dur5WK563HJuKYVfDwcuETx2p4ryS9vwij1/1tkwIQLkf1gEz8CYG1A0Q3GOawILQDAkZJQ+Ak/dpa6Js4OpC7jochEPeieWqTV7kUgZs5g1Bzg+MdSwTARGBsKtGajnRtc/hU8LOOwCglwAbZOCPbWK7KawFT1WQHQf1xBCEMaAvdIvaWguw/Q3NsvPuWHsQsN/EgEcKVwpwwHwoXcuq/GfiC7Gj1K8RIElwkF7EO6OjNwB0jovua70+lP9XtFUkqg7+oS1GfCdX230+eRIgYwafEQ6LVmSyAsXf/mOq2TNr3KcJh7oIAb6jrP13Do8H/UPnCgKQTlwkod2OXKTqyk+j5w0Qjqz5elMawZ1aoLFV8Zm8IFmTAO6c1jxHD86eYrjoULJNzgszOWWyHM3/2CQT+vy9UdyBEDkSE45NI2TFh4AztKpKueG4LOPplLVP8mjmmpujLCInyMjLMIpMpQicjkQBMrqkQjPOuQr8q3xVy1wK1WaCPJ25AKP0KQInTnrvEnlEPwTQixfIO6LDrbpTEAbaVe2q/dNo15iDfmfeBOGVCsa45cUFpL3SFQk/EM73Jd8eac3ROqQsjbmI0waiW2XHyrTjnfdSdULnCPqLNNFXx2afGqMo9EZTDT7VEFfd/VpI/QXQJOxeOvwQL8fe6ORjsL5vGaM4nJ+nBr3JqshiYzYeklmlfZi4magOyh4LzVXtuahMhRkQsk0czDH0YZpZ9KoYOKQlUw0f3ISGjxD5JWK+rQQtL6lTYE4oFF1qWaatOM0RGsQ0+BiidSMA2jISVeQZ6fKwDjd5mmiAtShLVN/kk59QImt/jm/Ix324mwxoyf8E54Fvu70OLhTktw4fgRJPPBRDsrS3bb6o3xSU3wYHRl5VQ8mTymU4h9qpqcIq6BITz9KbVivGN12SOhBk8NNSUzRQ2OWZPquzXScHTdwJ5Uc4HRQlNkLpTYoGZRvgYN4YA+TFhgL9xjoZxPywAFkMY7g5mN/NXVdCTRGFNQoSQMeijA69dskbsEcTHLwv2FB3RKAOGd4a8SoW7VnDNcbhWGWbEdGtcqH2D9wfkyzt8NwO8nIS+gOgOJPfWgNgeOnijNQ2/Vv/I4osV0uj3aUMMqn4KeV4rn0Xf8RHC3sUo3woY4kULCWrvRzcZoDFPXjkzZMBUa0cqxs4qjMb30klP2oYBCEDEUFhMklujOmGiw8Z66pKxbaPLCGxhWCIrxDRG+wUWCNm3qv8/Ar8a1jhqd7nqNh74sLhVOm9CqiaKGwGNtPbAOkNA53tsUTzcAguxS7moVjwp0gMxMW3UZn3dCVjpaqqfegbF45aA8Xhcy3WdqpqZrhVAq0mzEHWpiWa8KncdGdoPSLydgYwLoww2CXoauR+ijZrEuxjAGKUGObXThM17TgJaak85NTauQobNVS8NcYwzSRVDKNBk0JMPxa2XQylERh0c52tLyYaYfrAxZv5eqwmkn0I5OmdGpZpJetjDd4KKrz3yT2e4jIQCYIpA5Cy9OxkHlFn83AlNSVwmDa6KOqjfpb9BmjYfSZXJgMp1jDxykOFQe4rDLSDZU28cEr4aOUx6WsQlqhn3YUZq7k2c5cFkskVMHKNRzTcDUgqwWdaRdmEgJh0r8ZY2HIDI5xG6OKZ/Rahn0J2Wyp2PB5MpNyUvNRaWmU2h6tV6dn7IjGiDmBCDyZUrdLAihvu9fIJUChCUEZTbJ8XhpHwzEdsrsy2QCnHIrEJ/tEH4QYhhFdI/1ficAqboaWaIJWIWYckZKVGT+IEAAUjCviSNHBErGMrEKDPpbPIVi4TCBhURoJL0gYjCzNVNKDkvUBpDPJcTVeKzkJiZzSXF4ewdDpn+/1m4/kLWwi/+1UzvPrsqSG4OTSeaQ0dRIfvWmvexMR/Gb8YxV6LV2Co+fAnFe5yOb3MLQO2663eFOmVFq9yCREC8mY2qWyT44jo4J15aRQujSrxucCIQz1pqbVEPWxkj+0bHTxZNsUWh+WktyhrhQTiGhj6CbQpMGQfcPEyLu6IlGS74T0HmvImzruJ/m63yzf5TJiJQBBp5u+WJCPGsppDWL/x+2/ljHl6NWX1wC8E/tvZ2fD9187063vv+0y++8I6eeY7W1Gn4soNxbwKtuZkDRbCzyhSjaTXoo3tD0DmCJLTdJba8xhizJmfEP7ZCzWGyTtDhOSI1ppvpRZA7XUsUsgY+Ndv+FRicBMxowP78fNsXFEeLeEMROyGfR/1UsT0JHSjunKYaF7gVY0gBgk/Hw8/sgo+DkYV19/Yd3Grv7+3k3DYlYcQ/+ZVm0KyMmkDkOmfDF2RaCYAepXEY/j0ROfjXyaZXMkJnRcGrddI0y2LrtLpMSwN1U2UH8j9ZsGSr50jk4x8muYqZdt9vGhMsqQbIfFIRhoOHucTa9bMTfKKSF/PPKD0IMBy2xMGJmBThMgb7IMjMvl2bVlHH5SNScM0RXT0CeQPI2DM0RqjtMoksShSwtikBIFxkje3jwBcvUwsVq4TDQt3TFty+Cxy47jRF+Y4QtuOLeXzAu3Y3+4Zvb+4GMBTml8B8Z/xeMnqXq6x2LrN/UNysQU6DCQFCuSojMpVpKLAo6ApimYZ3Z4RDnIFcFgaD3V8axhTZ6cG9GZG/NPStnUX0TFFTxiVx6pS5zIkR9+gI1cxAzPp5BZ7FH+ar4VAmnQrEQ/UFAXqB2mC4644KtO2dJiYiPjlUGqgNmaJV3+auQDGN75Pt6+GXvJEsxF3DyGB32wXPbCivSYy2Qw6xz+m/eje5rE7QBMoSM65kfmcJ3gMfuLJ3cY+bJkSTHpW9Kxou8on9jW3FU+sU1ZT+niL9U9qVJE11iN9egKkESijKShPr7YlK2ZrD0U4sOF6w8GcOPmFCUz33SN21lIV5XRWCygeEG2Q57xpBpdhIFI+DjmT3RSRMOPKFJpLsgrydOebCxTl6o0LD85J82CyUh5P5ahQmkzgqajZeuLqp1wlTLqNI4CkJ+LRVaaAYGZogK5pboMKjL0CijC8g787EFFEvVZL3jeiS5cQqO4V3QOK1WBzksvKxwbucYpY+PeYca4e3uqWvaDHfv88Cy0Jlr2TDjoSCNhNpkmyV/V9uDfZg36E15Akh9P+u7PM+N2lSgMpqDjeIs+CkZzPZwjbEksDzPwxaRt1YhuCX9F6PgGvyKCLcAGmsDtDSAQs2QnNsumbsKpQhfIsDFsaTlGwi1V8yBOyiC+CXIRZZ/05gmC5cysoehBaZsFrs8OpeHURjpVHMGpt8b9hN6K0LUKphLGRNqtb/exfe95T9k1lPlhGWeZZQ6U4XGqEYOY9DURS8RUsYRJsX8bpPyT7H2xy/IKkGKlUlIcA/8PvEmpi+Zu0aFLbvf64Qynocuo807FtyVJOjHQiobzz+NDwihjkXgWNLb2SSnDpPgB0MfK4VHNX6bLdb6pX3Wj+/Uu8LQORcNQivDDEBQ/U5SHUiOG5dpORLRg8Zh2DPldqqJxBrBX8iBVAkFnMHpOLTfZZdKe51FvfDSxbhzQzqhijkJX+MVzE3nOZYyWWdxFFAVLLIuOkfIgqs+iBZQTmgbZqKUeoqa2TCjfXkoGPpkR7pecV6yYL8pwFnekRYc3UPVUw7Fd+d0T2J1oM8zCYxprZwcbpCySNK1ghOxsOhsGsmXykou6/huKhwuklXyQpHdWyim5UGQJX3pq7B7QaH/C5gnjkyf9VpjJNYhDTnKeBvLbliK2ZFzN/GRxgjwVNQKP4Lpg+uCguyGDZhL93CPe9E2j9ctNGg0OVXJlP08K/kzCg1I+tGBC+x7D5HsZZJr6zf7JoMGSSCEiH+SuQZNTw/MkEXJP+qjINGLpWCQvhbpAYTe4IXwpiBJC5GTUdZ6/ByZ9XUPxJ5/zt0G4O4JO0bxV10DsJcxCsBvnnQ0Gk2WRcC7aX3LjtJ131Hoo6tQ3r8wQiy0SqbT6PGrLaA4SsBB1byIRsBj+I62D0DIJMSqWkloUi+z0uhxsYS1yZg4LW3RUD7JUUlrjiQcp0o0yzapaIjRVGlGnK9RpmuxoGTwSZe3UOw/1NLGVwNiWaMQKHgcCrMy2L3JJCVam8xKNKbp8fZLzIQeNTiQ5V7Akw+SslMdMX9wSsV/+uSxS4kgRtrrjRFVNc1RxQbJEKY0uZViOFXWZzxqp1vyZqxrEgFKY3rf+gl3hy4fRKUUwYpHP5SBjJtc2MmNg/BGIPKUJLSdXpCljyQcxJ2QEczULE3WHskB4djsv5w/lxVW++sltUYdooNOa6KEJ/GwFmqwm9bkgboaV5jVK4mz67AS7KQScuWxMm/ThLJR/EeAUlIJxgQiJkU4pfs+5qCmJroVZCEPxJjB3JXJ4ppBPebbGdUIHGOXM1kabnUaQcUYCFWu4A/g3l46Y+OEJulJiXllKcL8qgQV8V5B+sozzkV0wl2iNQ6gyO7dVJiOWSjGCWqX84ADbSRe7bNxLhg2Q2BOT39jyjYB6EY2MbI2lw1JQgMamJvFbIOzEyXZKXMiRLjordkx86+44l3LdJsqGP+u9KqgkH8eHUGXrG3jJPgU8Wtqpll1A23E9eQqCLkLrCqWu4d0/L0zcXwpmBxl1IiFdBlpO/zs4/+HvdO3K85/0ipWrm5z/dLbOf3bR+U+Po89IRzUyMKOBAg8aXHIvmdTHjFSsm4zJbg0PJEDPXAfag4NOzxwFX0WOq23uG0FDJX0/flPfut7+4V7Ktc/vU5PIPkAVdACDDXTj8Gy76i7HQ+PCpA647GHiiVRo+q9MWSUq45vvS5ju7U+LAKDCg32bgk/F4HCS/AfH3eyUYAM5AHYND/eODPue9gV2ai8kkG0hpV0u4jzUuG/6lR+wC93J+UsAnmS8Y45Z/lMj19h/IAxfUyPu6hyile82QkdBeF18STyo6acxJBWwqUP2PI3AGDD9EadDY9fOYxSnkZq0nCSivgz8JTnbna9VzBnvp9wcN3T62mLcy25lj/Zt8isdYqvimZDOb8uo9+2o3bLnw1bF7yEsxvdQLGIsj7/LkoF7OeyYQi3nC+VDkCmFz1refWxBYzzQPNgT0WRo7EdN2ytF/+YjyQ22YopxYzIjS6DJUKskUsDEgf97WnxoQ3dXnOI5zFMNjdpHz7UqWju5Sn9v11Dv8AjDiAKv4I2aGqvaRS2+JFUDqTM8j+DvMaohDgHMMN+rVhDgYWanQK7RNibFiUnRmWACgMYgzN425YL04kgJJDhkCu15PC9JKpSNzlOCu0YdoEMG5WNkMo/KMmZo8KDz8L6hzFRLBE8AtrZU1D+nrw8dnRQdHfCfvx+RmvitIn6n8B8NulLcNmtT7MicJFxSticUA/SGBdIk4CyDQz+lOlicEIZIUdlLXRh1AHkz28n8RUjsm0JHQA7w36pht6uhewQHluZIsRI1KYyjoZHU2yxD8pHD9020FWyezFPzNREi9rz0JtIWh3Gojf2plIebgXeFOAwurhfjuTANY74eTEqOSHQ3WCOoRX+W4eFtdLngZwdFsDohLaPvTVSr2fjMOMASx8iF0rjtZOOODjcV2/IKuC5Zf/N2/k8IHw2KbZHhsGqR0Uj0QiLYNQ/tm4fWyQRcPuFczj/qdF0u1VifSGhjb/RYhj4rc4wYMmOWQV5HNObpxV8cFuxNufNN5uRkZDKx3hLRocynJJCKlZaRltVcfdoojJkVjV33pQdoSLbyHEimBrtAb+HXklbIpSBaRl6GTHaQ93NnFYWgcid5pQY6AiIMFLiWZHAAabLsyM1G+gIXzcoa7RkAqgwaI0rV2XYQa3GwABOoo7Zdy6ZfIXoIehwud0Hyqb7sn319qfJ/56rVq+rk//TK1a38X7tK/u+d1XH5yRlSDKEkZqIPTQ0DEuGZZ4O8Ak/YzQg/ygnPHSxW/1lPJeZL970aUrE+ikYpGyCuE32szgneb0GsMQoV5epcqklOrNiO0nxFYnkDO41qoWkwcyg0eRFF1OHxFiZG9b9o68lPprJxkiiiwABpzObD38fE02wL04PhAXKYQbWmz2/yaNKpETjEYfvoPufx97npS5++QcoKbEpaetnKRCqmNHQiRW/qk6LPjxnFezPqEHA8TEfwXRiM7Qy+ebmm4ZuX4VjQvI/l4G+J3N0D43NoQ0MbPZbBH0ohslZHwTTlsZNb90E+AAeN9MfTy/Ckwoz4OFEkbOi9oKk5gBMRiH2jPokLfNQixQ31tIXOVSL9ZeuOCiPQ91XqPrbrkrnKP91g6Y2mv4+MouQeoVvhR3+T/Mi2lK2yiHcu4FJV+bqg/9Vg4FqsY/khc0qMnAyMC7h62CaeGKDlE70u8GP2tIJoXOd18j8YIFd0mb9s0Q9o+G0cK4Me5OkMjQ9P//xZxoApLhB+iXEHIUDo9aZO6gDOzgwmRgGNqjTBlk78BJ9lmGPjRfiFeBZ+rRfT0/AyWIbvjUK+JOhdkVIiCEBoDJLD8iIr+U2bbAYlpY5jGIW6XKdBvlMZ/6N+xpS+fRkZsvoNU/l11Kbv2d9xBwX4Y6fye6pNS5BXIRVRS+QbMuUQopNfFm2Gpf4I5CZYykOpF/kjuVeDDNkZzmIY+IDtNFU8KQ8+sfGPzgryU2PUFj74n7SVxVSzdeQnkan/9ONLazlc9yBdrO8YF+7XzeTUa4UZdeirIMEnSSmnAzp1yph5psJWBOXhTdhVU1yRKbpAXDZLhhYW3mFcV8t+3bpaV+tqXa2rdbWu1tW6Wlfral2tq3W1rtbVulpX62pdrat1ta7W1bpaV+tqXa2rdbWu1tW6Wlfral2tq3W1rtb113b9N+GOy1IA8AAA"

try:
    import gymnasium
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gymnasium"], check=True)
shutil.rmtree("bombcompetition", ignore_errors=True)
shutil.rmtree("bots", ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ENGINE))) as tar:
    tar.extractall(".", filter="data")
if "" not in sys.path:
    sys.path.insert(0, "")
for name in [m for m in sys.modules if m.split(".")[0] in ("bombcompetition", "bots")]:
    del sys.modules[name]
importlib.invalidate_caches()

import torch, bombcompetition
from IPython.display import Image as _GifImage, display
from PIL import Image as _PIL
from bombcompetition import RULES, Game, SafeBot, legal_actions, random_rules
from bombcompetition.neural import CHANNELS, IN_CHANNELS, NeuralPolicy, preprocess, turn, turn_action
from bombcompetition.render import to_rgb
from bots.simple import Policy as _Simple


class SimpleBot(_Simple):
    """bots/simple.py"""


class RandomBot:
    """A random action that is not blocked."""
    def act(self, obs):
        return int(random.choice(legal_actions(obs)))


def play_game(bot_a, bot_b, rules=None, seed=None):
    """One game: bot_a plays A, bot_b plays B (Policy classes, or functions that make one), on `rules`
    (default: a new random map). Returns (game, actions, bots); game.rules, game.seed and the actions redraw it later."""
    seed = random.randrange(1 << 31) if seed is None else seed  # draws the hidden bomb fuses
    bots, game, actions = [SafeBot(bot_a), SafeBot(bot_b)], Game(rules or random_rules(), seed), []
    game.seed = seed
    while not game.done:
        a = (bots[0].act(game.obs(0)), bots[1].act(game.obs(1)))
        actions.append(a)
        game.step(*a)
    return game, actions, bots


def game_gif(actions, rules=RULES, seed=None, fps=8, tile=40) -> bytes:
    """Replay a game from its actions, map and seed (the engine is deterministic given them) as an animated GIF.
    The number on a bomb is its real fuse, which the players do not see."""
    game = Game(rules, seed)
    frames = [to_rgb(game, tile)]
    for a in actions:
        game.step(*a)
        frames.append(to_rgb(game, tile))
    frames += [frames[-1]] * 8  # hold the end for a moment
    imgs = [_PIL.fromarray(f).convert("P", palette=_PIL.ADAPTIVE, colors=64) for f in frames]
    buf = io.BytesIO()
    imgs[0].save(buf, format="GIF", save_all=True, append_images=imgs[1:], duration=1000 // fps, loop=0, optimize=True)
    return buf.getvalue()


def show_game(actions, rules=RULES, seed=None):
    display(_GifImage(data=game_gif(actions, rules, seed), format="gif"))


print("engine 551f8ad1 |", "torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Our code

The next cell unpacks our training code into the folder `bomb_repro/` (it is embedded in the cell, like the engine
above) and copies the engine there, so that folder is self-contained:

| Path in `bomb_repro/` | What it is |
|---|---|
| `arena/core.py`, `arena/grade.py`, `arena/panel.py` | local grading harness: plays bots and networks with the reference engine, the way the server does |
| `arena/nets.py` | the network definition (one source for training and for `submission.py`) and the export |
| `arena/rl/vecenv.py`, `arena/rl/ppo.py` | multi-process environments on the reference engine and the PPO league trainer |
| `arena/zoo/*.py` | opponent bots (section 4) |
| `runs/students/*` | student DQN networks (from the assets file) |
| `assets/best/` | the trained weights (from the assets file) |

Our code is only ever run in subprocesses inside `bomb_repro/` (`arena(...)` below); nothing of it is imported
into this notebook's kernel.

In [ ]:
#@title Unpack our code and the assets { display-mode: "form" }
import base64, hashlib, io, json, os, shutil, signal, subprocess, sys, tarfile, time
from pathlib import Path

CODE = """
...
"""
WORK = Path("bomb_repro").resolve()   # our code, the opponents and the training runs live here
ASSETS = "bomb_best_assets.tar.gz"    # the trained weights and the student networks

WORK.mkdir(exist_ok=True)
shutil.rmtree(WORK / "arena", ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))) as tar:
    tar.extractall(WORK, filter="data")
for d in ("bombcompetition", "bots"):  # the engine from the Install cell
    shutil.rmtree(WORK / d, ignore_errors=True)
    shutil.copytree(d, WORK / d, ignore=shutil.ignore_patterns("__pycache__"))

assets = next((p for p in (Path(ASSETS), WORK / ASSETS, Path("/content") / ASSETS) if p.exists()), None)
if assets is None:
    try:
        from google.colab import files
        print(f"Upload {ASSETS} now (cancel to go on without it).")
        uploaded = files.upload()
        assets = Path(next(iter(uploaded))) if uploaded else None
    except ImportError:
        pass
if assets is not None:
    with tarfile.open(assets) as tar:
        tar.extractall(WORK, filter="data")

HAVE_BEST = (WORK / "assets/best/submission.pth").exists()
STUDENTS = sorted(p.name for p in (WORK / "runs/students").glob("train_*") if (p / "submission.pth").exists())
CPUS = os.cpu_count() or 2


def arena(*args):
    """Run `python -m <module> ...` inside bomb_repro/ and show its output, e.g. arena("arena.grade", "net:...", "--panel", "val")."""
    r = subprocess.run([sys.executable, "-m", *map(str, args)], cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout)
    if r.returncode:
        raise RuntimeError(f"{args[0]} failed with exit code {r.returncode}")


print(f"code: {len(list((WORK / 'arena').rglob('*.py')))} files in {WORK / 'arena'}")
print(f"assets: {'found ' + str(assets) if assets else 'NOT FOUND'} | trained weights: {HAVE_BEST} | student networks for training: {len(STUDENTS)}")

## 3. Submission cell

**This cell becomes `submission.py`.** It is the file `arena/nets.py` generates for a 64-channel, 6-block network
(`arena.nets.render_source`), so the class below is exactly the one the trainer optimises.

- Learned layers only: a 3x3 stem, 6 pre-activation residual blocks with GroupNorm, and a small head that reads the
  tower at my cell and at the opponent's cell (the two input planes that mark them) plus global mean and max
  pooling. No rules, no search.
- No BatchNorm or Dropout, so the scores are the same in `train()` and `eval()` mode.
- `forward` returns the 6 action scores (the policy logits). The value head `v` is only used during training.

In [ ]:
%%writefile submission.py
# Everything grading needs is in this file. The server has: numpy, torch (CPU), bombcompetition.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from bombcompetition.neural import CHANNELS, IN_CHANNELS  # the fixed input: (IN_CHANNELS, 9, 9)

CFG = {'channels': 64, 'blocks': 6, 'norm': 'gn', 'head': 'gather', 'head_hidden': 256, 'sym': 1}
RETURN_VALUE = False  # training: (scores, value); submission: the 6 action scores only

_ACTION_DELTAS = ((0, 0), (-1, 0), (1, 0), (0, -1), (0, 1), (0, 0))  # STAY UP DOWN LEFT RIGHT BOMB
# The 8 symmetries of the board as 2x2 matrices on (row, col) offsets from the centre; 0 = identity, 1 = transpose.
_SYMMETRIES = (((1, 0), (0, 1)), ((0, 1), (1, 0)), ((-1, 0), (0, -1)), ((0, -1), (-1, 0)),
               ((-1, 0), (0, 1)), ((1, 0), (0, -1)), ((0, 1), (-1, 0)), ((0, -1), (1, 0)))


def _symmetry_tables(n=9):
    """cell[k][i] = the source cell of cell i in the board seen through symmetry k;
    act[k][a] = the action that action a becomes in that view."""
    c = (n - 1) // 2
    cells, acts = [], []
    for m in _SYMMETRIES:
        src = [0] * (n * n)
        for r in range(n):
            for q in range(n):
                r2 = m[0][0] * (r - c) + m[0][1] * (q - c) + c
                q2 = m[1][0] * (r - c) + m[1][1] * (q - c) + c
                src[r2 * n + q2] = r * n + q
        cells.append(src)
        act = []
        for a, (dr, dc) in enumerate(_ACTION_DELTAS):
            d2 = (m[0][0] * dr + m[0][1] * dc, m[1][0] * dr + m[1][1] * dc)
            act.append(a if a in (0, 5) else _ACTION_DELTAS.index(d2, 1, 5))
        acts.append(act)
    return cells, acts


def _norm(kind, c):
    if kind == "gn":
        return nn.GroupNorm(8, c)
    if kind == "ln":
        return nn.GroupNorm(1, c)
    return nn.Identity()


class Block(nn.Module):
    """Pre-activation residual block; the last conv starts at zero, so every block starts as the identity."""

    def __init__(self, c, kind):
        super().__init__()
        self.n1, self.c1 = _norm(kind, c), nn.Conv2d(c, c, 3, padding=1)
        self.n2, self.c2 = _norm(kind, c), nn.Conv2d(c, c, 3, padding=1)
        nn.init.zeros_(self.c2.weight)
        nn.init.zeros_(self.c2.bias)

    def forward(self, x):
        y = self.c1(F.relu(self.n1(x)))
        return x + self.c2(F.relu(self.n2(y)))


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        c, kind, hid = CFG["channels"], CFG["norm"], CFG["head_hidden"]
        self.sym = int(CFG["sym"])
        self.head = CFG["head"]
        self.stem = nn.Conv2d(IN_CHANNELS, c, 3, padding=1)
        self.blocks = nn.ModuleList([Block(c, kind) for _ in range(CFG["blocks"])])
        self.nf = _norm(kind, c)
        if self.head == "gather":
            feat = 4 * c
        else:
            self.squeeze = nn.Conv2d(c, 16, 1)
            feat = 16 * 81
        self.pi = nn.Sequential(nn.Linear(feat, hid), nn.ReLU(), nn.Linear(hid, 6))
        self.v = nn.Sequential(nn.Linear(feat, hid), nn.ReLU(), nn.Linear(hid, 1))
        cells, acts = _symmetry_tables()
        self.register_buffer("sym_cell", torch.tensor(cells, dtype=torch.long), persistent=False)
        self.register_buffer("sym_act", torch.tensor(acts, dtype=torch.long), persistent=False)

    def features(self, x):
        h = self.stem(x)
        for block in self.blocks:
            h = block(h)
        h = F.relu(self.nf(h))
        if self.head == "gather":
            me, opp = x[:, 1:2], x[:, 2:3]  # the input planes that mark my cell and the opponent's cell
            return torch.cat([(h * me).sum((2, 3)), (h * opp).sum((2, 3)), h.mean((2, 3)), h.amax((2, 3))], 1)
        return F.relu(self.squeeze(h)).flatten(1)

    def scores_value(self, x):
        z = self.features(x)
        return self.pi(z), self.v(z).squeeze(1)

    def forward(self, x):
        if self.sym <= 1:
            scores, value = self.scores_value(x)
        else:  # average over board symmetries: view k of the board, scores mapped back to the real actions
            b, k = x.shape[0], self.sym
            flat = x.flatten(2)
            views = torch.cat([flat[:, :, self.sym_cell[i]] for i in range(k)], 0).view(k * b, *x.shape[1:])
            s, v = self.scores_value(views)
            s = s.view(k, b, 6)
            scores = sum(s[i][:, self.sym_act[i]] for i in range(k)) / k
            value = v.view(k, b).mean(0)
        return (scores, value) if RETURN_VALUE else scores

## 4. Opponents: the league

Every training game draws its opponent from a weighted list; opponents that the agent does not beat yet are drawn
more often (prioritised sampling: weight x (0.1 + 0.9 x (1 - win rate))). All bots are in `bomb_repro/arena/zoo/`.

| Opponent | How it plays | v1 weight | v2 weight |
|---|---|---|---|
| `random` | a random legal action | 0.5 | 0.3 |
| `simple` | the course's SimpleBot | 0.5 | 0.3 |
| `zoo.base_cautious` | dodges with exact fuse timing, bombs when the opponent is in its blast, chases | 1 | 1 |
| `zoo.ta_smart_a` | our guess at "SmartBot": dodge, bomb when the opponent has at most one way out, leave the closing ring, BFS chase | 3 | 3 |
| `zoo.ta_smart_b` | aggressive: bombs whenever the opponent is in range, up to 3 bombs to cut escapes; plans its own escape as if the opponent stood still | 1.5 | 1.5 |
| `zoo.evader` | survival first: only actions that survive every opponent reply, avoids dead ends, bombs only when fully safe | 2 | 2 |
| `zoo.ta_final_a` | strong: resolves every (my action, opponent action) pair exactly, three levels of safety checks, forced-kill detection | 3 | 3 |
| `zoo.ta_final_b` | positional: bombs as temporary walls, squeezes the opponent's region, takes the centre before a shrink | 3 | 3 |
| `zoo.param` | the strongest preset of the parametrised family | 1 | 1 |
| `zoo.param:<seed>` | parametrised family: about 20 behaviour switches (what it knows about explosions, when it bombs, how it chases, how random it is, SimpleBot's freeze bug, ...) sampled from the seed | seeds 0-39, 0.2 each | seeds 0-199, 0.1 each |
| `zoo.x_solver` | search bot: forced-win search, keeps only actions that are safe against every reply, then maximum pressure | - | 2 |
| `noisy:<eps>:<bot>` | a strong bot that plays a random legal action with probability eps (teaches punishing mistakes) | 3 bots at 0.1, evader at 0.2 | 4 bots at 0.1, evader at 0.2, ta_smart_a at 0.25 |
| `net:runs/students/train_*` | 6 baseline DQN networks trained with the course notebook and typical student changes | - | 0.4 each |
| `self` | the current network on both sides (both sides are training data) | 3 | 3 |
| `past` | a frozen copy of the network from earlier in training (pool of 4) | 2 | 3 |

**Held out.** `zoo.val_smart`, `zoo.val_hunter`, `zoo.val_tricky` and `zoo.param:9000`-`9005` are only used to compare
runs and checkpoints; `zoo.test_smart`, `zoo.test_boss`, `zoo.test_camper` and the `test_*` student networks are only
used for the final report. None of them is ever a training opponent.

In [ ]:
# The two leagues as "spec:weight" lists (the trainer's --league format).
BOTS = ["zoo.base_cautious:1", "zoo.ta_smart_a:3", "zoo.ta_smart_b:1.5", "zoo.evader:2", "zoo.ta_final_a:3",
        "zoo.ta_final_b:3", "zoo.param:1"]

LEAGUE_V1 = ",".join(
    ["simple:0.5", "random:0.5"] + BOTS
    + [f"zoo.param:{s}:0.2" for s in range(40)]
    + ["noisy:0.1:zoo.ta_smart_a:1", "noisy:0.1:zoo.ta_final_a:1", "noisy:0.1:zoo.ta_final_b:1", "noisy:0.2:zoo.evader:0.5"]
    + ["self:3", "past:2"])

LEAGUE_V2 = ",".join(
    ["simple:0.3", "random:0.3"] + BOTS + ["zoo.x_solver:2"]
    + [f"zoo.param:{s}:0.1" for s in range(200)]
    + [f"noisy:0.1:{b}:0.7" for b in ("zoo.ta_smart_a", "zoo.ta_final_a", "zoo.ta_final_b", "zoo.x_solver")]
    + ["noisy:0.2:zoo.evader:0.5", "noisy:0.25:zoo.ta_smart_a:0.5"]
    + [f"net:runs/students/{s}:0.4" for s in STUDENTS]
    + ["self:3", "past:3"])

# graded after every EVAL_EVERY updates (argmax policy, 100 games each); never trained against
EVAL_OPP = "zoo.val_smart,zoo.val_hunter,zoo.val_tricky,zoo.param:9000,zoo.param:9001,zoo.param:9002,zoo.param:9003"

print(f"league v1: {LEAGUE_V1.count(',') + 1} opponents | league v2: {LEAGUE_V2.count(',') + 1} opponents"
      + ("" if STUDENTS else "  (WARNING: no student networks found, league v2 runs without them)"))

## 5. Training: two PPO stages

The trainer is `bomb_repro/arena/rl/ppo.py` (`python -m arena.rl.ppo --help` lists every option). In short:

- **Environments.** 2048 games in parallel on the reference engine (64 worker processes x 32 games), the learner's
  side chosen at random, a new random map every game. The learner sees `preprocess(turn(obs))` and samples from
  softmax(scores); a blocked move simply acts as STAY (no action masking, because the server takes a plain argmax).
- **PPO.** 64 steps per environment per update (131,072 steps per update), GAE (gamma 0.995, lambda 0.95),
  2 epochs, minibatches of 8192, clip 0.2, Adam with learning rate 3e-4 (5 warm-up updates, cosine decay to 10 %),
  entropy bonus 0.02 falling to 0.005, gradient clipping 1.0.
- **Reward.** Only at the end of a game: win +1; draw and loss as given per stage. No shaping.
- **Checkpoints.** Every 40 updates the network is exported and its argmax policy is graded on the validation bots
  (100 games each) in the background.

| | Stage 1 | Stage 2 |
|---|---|---|
| start | random weights, seed 0 | the stage-1 network after 200 updates, seed 3 |
| league | v1 | v2 |
| reward for draw / loss | -0.1 / -0.4 | -0.3 / -0.5 |
| schedules run over | 300 M steps | 120 M steps |
| stopped after | 200 updates (26.2 M steps) | 628 updates (82.3 M steps) |
| extra | | the first 3 updates train only the value head |

The submitted network is the stage-2 network after its last update (628). Stage 2 ran under a wall-clock limit in
the original run, which ended it there; `--stop-update` reproduces the same stopping point exactly.

In [ ]:
import torch

MODE = "restore"   # "restore": the trained weights from the assets file | "train": run both stages from scratch

# the original run (change these only for a quick test)
STAGE1_UPDATES, STAGE2_UPDATES = 200, 628
EVAL_EVERY, EVAL_GAMES = 40, 100
N_ENVS = 2048
GRADE_GAMES = 100                                   # games per opponent in section 6

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
WORKERS = min(64, max(1, CPUS - 2))                 # 64 in the original run
PROCS = max(1, min(96, CPUS - 1))                   # processes for grading
EXPECTED_MD5 = "e3e5edae202d88beeb291b4db92aa0f7"       # the submitted weights

PPO = ["--rollout", 64, "--epochs", 2, "--minibatch", 8192, "--gamma", 0.995, "--lam", 0.95, "--clip", 0.2,
       "--vf-coef", 0.5, "--max-grad-norm", 1.0, "--target-kl", 0.03, "--lr", 3e-4, "--lr-final", 0.1, "--lr-warmup", 5,
       "--ent-coef", 0.02, "--ent-coef-final", 0.005, "--pfsp-power", 1, "--pfsp-floor", 0.1, "--pfsp-decay", 0.9,
       "--snapshots", 4, "--snap-every", 25, "--snap-greedy", 0.5, "--reward-win", 1]


def evals(out):
    f = Path(out) / "evals.jsonl"
    return [json.loads(line) for line in f.read_text().splitlines()] if f.exists() else []


def run_stage(name, args, stop_update):
    """Run the trainer for exactly `stop_update` updates; it then exports <out>/final and grades it."""
    out = WORK / "runs" / "rl" / name
    meta = out / "final" / "meta.json"
    if meta.exists() and json.loads(meta.read_text()).get("update") == stop_update:
        print(f"{name}: already done ({out})")
        return out
    shutil.rmtree(out, ignore_errors=True)
    cmd = [sys.executable, "-u", "-m", "arena.rl.ppo", "--out", f"runs/rl/{name}", "--device", DEVICE,
           "--workers", WORKERS, "--envs-per-worker", max(1, N_ENVS // WORKERS), "--groups", 2 if WORKERS > 1 else 1,
           "--eval-opp", EVAL_OPP, "--eval-every", EVAL_EVERY, "--eval-games", EVAL_GAMES,
           "--eval-procs", max(1, min(12, CPUS // 4)), "--stop-update", stop_update, *PPO, *args]
    proc = subprocess.Popen(list(map(str, cmd)), cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            start_new_session=True)
    try:
        for line in proc.stdout:
            if not line.startswith("[ppo] u") or int(line.split()[1][1:]) % 20 == 0:
                print(line.rstrip()[:260 if line.startswith("[eval]") else 150], flush=True)
        if proc.wait() != 0 or not meta.exists():
            raise RuntimeError(f"the trainer failed (exit code {proc.returncode})")
    finally:
        if proc.poll() is None:  # interrupted: stop the trainer and its workers (they form one process group)
            for sig in (signal.SIGTERM, signal.SIGKILL):
                try:
                    os.killpg(proc.pid, sig)
                    proc.wait(timeout=30)
                    break
                except (ProcessLookupError, subprocess.TimeoutExpired):
                    pass
    return out


print(f"MODE = {MODE} | device {DEVICE} | {CPUS} CPUs -> {WORKERS} workers x {max(1, N_ENVS // WORKERS)} environments")

In [ ]:
# Stage 1: from random weights against league v1 (draw -0.1, loss -0.4).
if MODE == "train":
    STAGE1 = run_stage("stage1", ["--league", LEAGUE_V1, "--channels", 64, "--blocks", 6, "--norm", "gn", "--head", "gather",
                                  "--head-hidden", 256, "--reward-draw", -0.1, "--reward-loss", -0.4,
                                  "--total-steps", 300_000_000, "--seed", 0], STAGE1_UPDATES)
else:
    print("MODE = restore: nothing to train")

In [ ]:
# Stage 2: continue from the stage-1 network against league v2 (draw -0.3, loss -0.5).
if MODE == "train":
    STAGE2 = run_stage("stage2", ["--league", LEAGUE_V2, "--init", "runs/rl/stage1/final", "--value-warmup", 3,
                                  "--reward-draw", -0.3, "--reward-loss", -0.5, "--total-steps", 120_000_000, "--seed", 3],
                       STAGE2_UPDATES)
    for e in evals(STAGE2):  # the validation history (100 games per opponent, so it is noisy)
        print(f"update {e['update']:>4}: validation mean win rate {e['mean_win_rate']:.3f}, loss rate {e['mean_loss_rate']:.3f}")
    FINAL = STAGE2 / "final"  # the network after the last update
else:
    assert HAVE_BEST, "MODE = restore needs bomb_best_assets.tar.gz next to the notebook (see section 2)"
    FINAL = WORK / "assets" / "best"
print("final network:", FINAL)

In [ ]:
# Load the final weights into the Net of the submission cell (strict: the names and shapes must match).
import importlib

import submission
importlib.reload(submission)  # picks up the submission cell
from submission import Net

net = Net()
net.load_state_dict(torch.load(FINAL / "submission.pth", map_location="cpu"))
net.eval()
md5 = hashlib.md5((FINAL / "submission.pth").read_bytes()).hexdigest()
print(f"{sum(p.numel() for p in net.parameters()):,} parameters | weights md5 {md5}"
      + (" (= the submitted network)" if md5 == EXPECTED_MD5 else " (a retrained network)"))
if MODE == "restore":
    assert md5 == EXPECTED_MD5, "the assets file does not hold the submitted weights"
# the trainer exported the same class: the two files must be the same network definition
same = (FINAL / "submission.py").read_text().strip() == Path("submission.py").read_text().strip()
print("submission cell == the trainer's exported submission.py:", same)

## 6. Local evaluation

First the course's check against SimpleBot, then our own grading, which plays the network exactly as the server
does (`NeuralPolicy`, argmax) against the held-out bots: `GRADE_GAMES` games per opponent, every map from both
sides, maps that were never used in training. Columns: W D L, `self` = games we lost to our own bomb,
`oppself` = opponent self-bombs, `t/o` = draws by the 300-step limit.

In [ ]:
# The course's local evaluation (section 6 of the template), against SimpleBot.
random.seed(0)
total = {"win": 0, "draw": 0, "loss": 0}
for i in range(20):
    side = i % 2
    mine = lambda: NeuralPolicy(net)  # exactly how the server plays the network
    pair = (mine, SimpleBot) if side == 0 else (SimpleBot, mine)
    game, actions, bots = play_game(*pair, random_rules(seed=1000 + i // 2))
    w = game.winner()
    total["win" if w == side else "draw" if w is None else "loss"] += 1
print(f"vs SimpleBot: {total['win']}W {total['draw']}D {total['loss']}L")

In [ ]:
# Held-out validation bots (used to choose the checkpoint) and test bots (not used for any decision).
arena("arena.grade", f"net:{FINAL}", "--panel", "val", "--games", GRADE_GAMES, "--procs", PROCS)
arena("arena.grade", f"net:{FINAL}", "--panel", "test", "--games", GRADE_GAMES, "--procs", PROCS)

## 7. Make the submission

The course's last cell, unchanged: it saves `submission.pth` from `net`, plays one game in a fresh Python process in
a folder that holds only the two files, and builds `submission.zip`.

In [ ]:
import os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

torch.save({k: v.detach().cpu() for k, v in net.state_dict().items()}, "submission.pth")

CHECK = """
import importlib.util
import torch
from bombcompetition import RULES, Game, SafeBot, random_rules
from bombcompetition.neural import NeuralPolicy
spec = importlib.util.spec_from_file_location("submission_check", "submission.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

def server_policy():  # what the server does
    net = module.Net()
    net.load_state_dict(torch.load("submission.pth", map_location="cpu"))
    return NeuralPolicy(net.eval())  # the fixed preprocess

bot = SafeBot(server_policy)
assert bot.policy is not None, "Net() or loading submission.pth failed: " + "; ".join(bot.errors)
game = Game(random_rules(0))
while not game.done:
    game.step(bot.act(game.obs(0)), 0)
assert bot.faults == 0, "Net failed: " + "; ".join(bot.errors)
print(f"Check passed: one game with no faults (slowest act {bot.slowest * 1000:.1f} ms)")
"""
with tempfile.TemporaryDirectory() as tmp:
    for f in ("submission.py", "submission.pth"):
        shutil.copy(f, tmp)
    result = subprocess.run([sys.executable, "-c", CHECK], cwd=tmp, capture_output=True, text=True,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONPATH": os.getcwd()})
print(result.stdout, result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit("Check failed: fix the error above, then re-run the submission cell and this cell.")

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ("submission.py", "submission.pth"):
        zf.write(f)
print("submission.zip:", ", ".join(f"{i.filename} ({i.file_size:,} B)" for i in zipfile.ZipFile("submission.zip").infolist()))

try:
    from google.colab import files
    files.download("submission.zip")
except ImportError:
    print("Not on Colab, so no automatic download:", Path("submission.zip").resolve())

In [ ]:
# The zip holds exactly the network that was graded above.
import zipfile

with zipfile.ZipFile("submission.zip") as zf:
    state = torch.load(io.BytesIO(zf.read("submission.pth")), map_location="cpu")
ref = torch.load(FINAL / "submission.pth", map_location="cpu")
assert state.keys() == ref.keys() and all(torch.equal(state[k], ref[k]) for k in ref)
print("submission.zip is ready:", Path("submission.zip").resolve())